# Interview Transcriber — launcher

Use a T4 GPU runtime and the same Google account whose Drive contains the recordings. Store the Hugging Face read-only token once as `HF_TOKEN` in Colab Secrets. Run the single START cell below. It downloads the current launcher bootstrap from `main` with a GitHub API fallback, so future launcher fixes are picked up automatically.

In [ ]:
# @title ▶ START INTERVIEW TRANSCRIBER
import base64
import json
import time
import urllib.request

cache_buster = time.time_ns()
raw_url = (
    "https://raw.githubusercontent.com/playply/transcriber/main/colab_bootstrap.py"
    f"?cb={cache_buster}"
)
api_url = (
    "https://api.github.com/repos/playply/transcriber/contents/colab_bootstrap.py"
    f"?ref=main&cb={cache_buster}"
)

print("Fetching current launcher...", flush=True)
source = None
errors = []

for label, url in (("GitHub raw", raw_url), ("GitHub API", api_url)):
    try:
        print(f"  trying {label}...", flush=True)
        request = urllib.request.Request(
            url,
            headers={
                "Cache-Control": "no-cache",
                "Pragma": "no-cache",
                "Accept": "application/vnd.github+json",
                "User-Agent": "interview-transcriber-colab",
            },
        )
        with urllib.request.urlopen(request, timeout=15) as response:
            payload = response.read()

        if label == "GitHub API":
            data = json.loads(payload.decode("utf-8"))
            source = base64.b64decode(data["content"]).decode("utf-8")
        else:
            source = payload.decode("utf-8")

        if "LAUNCHER_BUILD" not in source:
            raise RuntimeError("downloaded content is not the launcher bootstrap")

        print(f"✓ Launcher downloaded via {label}", flush=True)
        break
    except Exception as exc:
        errors.append(f"{label}: {type(exc).__name__}: {exc}")
        print(f"  {label} failed; trying fallback...", flush=True)

if source is None:
    raise RuntimeError(
        "Could not download the launcher from GitHub.\n" + "\n".join(errors)
    )

exec(compile(source, "colab_bootstrap.py", "exec"), {"__name__": "__main__"})
